# 07 — Multilingual tokenization benchmark (Part G)

Implement `src/evaluation.py` (`pytest tests/test_evaluation.py`), then compare **GPT-2, BERT, FLAN-T5, your BPE and your WordPiece** on 300 evaluation sentences (100 English, 100 French, 50 MSA, 50 Algerian Darija).

Rules: same sentences for every tokenizer · no special tokens counted · metrics from the *totals* · evaluation sentences are never used for training.

In [ ]:
import sys, pathlib, json, time
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
%load_ext autoreload
%autoreload 2

from src.pretrained import load_pretrained
from src.tokenizer_io import load_tokenizer
from src.evaluation import TokenizerAdapter, evaluate_sentences, summarize, time_encoding, parity
eval_df = pd.read_csv(ROOT / "data/eval/eval_sentences.csv")
print(eval_df.language.value_counts().to_dict())
eval_df.groupby("language").sample(2, random_state=0)

## 1. Collect the five tokenizers

In [ ]:
tokenizers_ = {}
for name, tok in load_pretrained().items():
    tokenizers_[{"gpt2": "GPT-2", "bert-base-uncased": "BERT", "google/flan-t5-small": "FLAN-T5"}.get(name, name)] = tok
tokenizers_["Custom BPE"] = load_tokenizer(ROOT / "models/bpe/tokenizer.json")
tokenizers_["Custom WordPiece"] = load_tokenizer(ROOT / "models/wordpiece/tokenizer.json")
print({n: type(t).__module__.split(".")[0] + "." + type(t).__name__ for n, t in tokenizers_.items()})

## 2. Warm-up: two APIs, one question — *how many tokens?*

Your custom tokenizers are `tokenizers.Tokenizer` objects; the pretrained ones are Transformers tokenizers. This is the problem the `TokenizerAdapter` solves.

### 🧩 Exercise 7.1 — Count tokens through both APIs

Count the tokens of one sentence with a raw tokenizer **and** with a Transformers tokenizer, **without** special tokens in both cases.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* raw `tokenizers.Tokenizer`: `tok.encode(text, add_special_tokens=False).tokens`.
* Transformers: `tok.tokenize(text)` (no special tokens) — or `tok.encode(text, add_special_tokens=False)` for ids.
* Why `add_special_tokens=False`? (notebook 02, exercise 2.2)

In [ ]:
sentence = eval_df.text[0]
raw_tok = tokenizers_["Custom BPE"]
hf_tok = tokenizers_["BERT"] if "BERT" in tokenizers_ else list(tokenizers_.values())[0]

n_raw = len(raw_tok.encode(sentence, ____).____)
n_hf  = len(hf_tok.____(sentence))
print(sentence); print("raw API:", n_raw, "tokens | Transformers API:", n_hf, "tokens")

In [ ]:
assert n_raw == len(raw_tok.encode(sentence, add_special_tokens=False).ids)
assert n_hf == len(hf_tok.encode(sentence, add_special_tokens=False))
print("✅ Exercise 7.1 OK")

## 3. Sentence-level results → CSV

In [ ]:
adapters = {name: TokenizerAdapter(name, tok) for name, tok in tokenizers_.items()}
print({n: a.vocab_size for n, a in adapters.items()})
results = pd.concat([evaluate_sentences(a, eval_df) for a in adapters.values()], ignore_index=True)
results.to_csv(ROOT / "results/sentence_level_results.csv", index=False)
print(len(results), "rows"); results.head()

### 🧩 Exercise 7.2 — Compute the metrics by hand with pandas

On the sentence-level table of **one** tokenizer, compute: average tokens, characters per token (ratio of totals!), tokens per word and unknown-token rate. Then do the same per language with `groupby`.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* Columns: `n_tokens`, `n_chars`, `n_words`, `n_unk`. Series methods: `.mean()`, `.sum()`.
* characters per token = total characters / total tokens (**not** the mean of per-sentence ratios).
* `df.groupby("language")["n_tokens"].mean()` gives one value per language.
* Your `summarize()` must produce the same numbers: the check compares them.

In [ ]:
df = results[results.tokenizer == "Custom BPE"]

avg_tokens      = df["n_tokens"].____()
chars_per_token = df["n_chars"].____() / df["n_tokens"].____()
tokens_per_word = df["n_tokens"].sum() / df["n_words"].____()
unk_rate        = df["n_unk"].sum() / df["n_tokens"].sum()

per_language = df.____("language")["n_tokens"].mean()
print(round(avg_tokens, 2), round(chars_per_token, 3), round(tokens_per_word, 3), unk_rate)
print(per_language.round(2).to_dict())

In [ ]:
ref = summarize(results).set_index("tokenizer").loc["Custom BPE"]
assert abs(ref["avg_tokens"] - avg_tokens) < 1e-9
assert abs(ref["chars_per_token"] - chars_per_token) < 1e-9
assert abs(ref["tokens_per_word"] - tokens_per_word) < 1e-9
assert abs(ref["unk_rate"] - unk_rate) < 1e-9
print("✅ Exercise 7.2 OK  (your hand computation agrees with src/evaluation.summarize)")

## 4. Comparison tables

### 🧩 Exercise 7.3 — Time a tokenizer properly

Measure how long one adapter needs to encode all evaluation sentences, **after one warm-up pass**, averaged over several repetitions.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `time.perf_counter()` returns a high-resolution clock (seconds); take the difference before/after.
* The first pass is often slower (caches, lazy initialisation): do one untimed warm-up pass.
* Average over `repeats` timed passes. (`src.evaluation.time_encoding` does exactly this; here you write the core yourself.)

In [ ]:
adapter = adapters["Custom BPE"]
texts = list(eval_df.text)

for s in texts: adapter.encode(s)                    # warm-up (not timed)
times = []
for _ in range(5):
    t0 = time.____()
    for s in texts:
        adapter.encode(s)
    times.append(time.____() - t0)
mean_time = ____
print(f"{mean_time * 1000:.1f} ms per pass over {len(texts)} sentences")

In [ ]:
assert 0 < mean_time < 60 and len(times) == 5
print("✅ Exercise 7.3 OK")

In [ ]:
vocab_sizes = {n: a.vocab_size for n, a in adapters.items()}
overall = summarize(results, vocab_sizes=vocab_sizes)
overall["encode_time_s"] = [time_encoding(adapters[n], eval_df.text, repeats=5) for n in overall.tokenizer]
overall.to_csv(ROOT / "results/comparison.csv", index=False)
overall.round(3)

In [ ]:
by_lang = summarize(results, group_cols=["tokenizer", "language"], vocab_sizes=vocab_sizes)
by_lang.to_csv(ROOT / "results/comparison_by_language.csv", index=False)
for lang in ["en", "fr", "ar_msa", "ar_darija"]:
    print("\n==", lang); print(by_lang[by_lang.language == lang].drop(columns="language").round(3).to_string(index=False))

### Read the `unk_rate` column BEFORE interpreting token counts

A tokenizer that gives up on many words can have very few tokens. Always report both.

## 5. Fairness across languages: parity

In [ ]:
par = parity(by_lang, reference="en", metric="tokens_per_word")
par.round(2)

## 6. Figures

### 🧩 Exercise 7.4 — Reshape a table for a grouped bar chart

Turn the long table `by_lang` (one row per tokenizer × language) into a wide table (rows = tokenizer, columns = language) and plot it as grouped bars.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `df.pivot(index=..., columns=..., values=...)`.
* `.loc[order]` fixes the row order so every figure uses the same tokenizer order.
* `DataFrame.plot(kind="bar")` draws one group per row.

In [ ]:
order = list(adapters)
pivot = by_lang.____(index="tokenizer", columns=____, values="avg_tokens").loc[order]
ax = pivot.____(kind="bar", figsize=(9, 4))
ax.set_ylabel("average tokens per sentence"); ax.set_title("Average sequence length by language")
plt.xticks(rotation=20); plt.tight_layout(); plt.savefig(ROOT / "results/figures/avg_tokens_per_language.png", dpi=150); plt.show()

In [ ]:
assert pivot.shape == (len(order), 4) and list(pivot.index) == order
print("✅ Exercise 7.4 OK")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), sharey=True)
for ax, lang in zip(axes, ["en", "fr", "ar_msa", "ar_darija"]):
    data = [results[(results.tokenizer == n) & (results.language == lang)].n_tokens for n in order]
    ax.boxplot(data); ax.set_xticklabels([n.replace("Custom ", "C-") for n in order], rotation=60); ax.set_title(lang)
axes[0].set_ylabel("tokens per sentence"); plt.tight_layout(); plt.savefig(ROOT / "results/figures/length_distribution.png", dpi=150); plt.show()

In [ ]:
cpt = by_lang.pivot(index="tokenizer", columns="language", values="chars_per_token").loc[order]
ax = cpt.plot(kind="bar", figsize=(9, 4)); ax.set_ylabel("characters per token"); ax.set_title("Characters per token (higher = more compact)")
plt.xticks(rotation=20); plt.tight_layout(); plt.savefig(ROOT / "results/figures/chars_per_token.png", dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
v = overall.set_index("tokenizer").loc[order]
ax[0].bar(order, v.vocab_size); ax[0].set_title("Vocabulary size"); ax[0].tick_params(axis="x", rotation=30)
u = by_lang.pivot(index="tokenizer", columns="language", values="unk_rate").loc[order]
u.plot(kind="bar", ax=ax[1]); ax[1].set_title("Unknown-token rate"); ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(ROOT / "results/figures/vocab_and_unk.png", dpi=150); plt.show()

## 7. Analysis (use the numbers above; claim → number → mechanism → caveat)

1. Which tokenizer generates the fewest tokens, overall and per language? Is it the "best"? (check `unk_rate`)
2. Does a larger vocabulary necessarily give fewer tokens? Use at least two tokenizers.
3. Which tokenizer is the most efficient for French? 4. For Arabic and for Darija? Is Darija more or less costly than MSA, and why might that be?
5. Why does a tokenizer trained mainly on English/French behave badly on Arabic script? Distinguish the *byte-level* case and the *`[UNK]`* case.
6. What does the vocabulary size change?
7. Is the tokenizer with the lowest average token count necessarily the best? List at least three other criteria.
8. Encoding time: how do you explain the differences? What are the limits of a single-sentence timing loop?

**Your answer:**

_(write here)_

## 8. Limitations (for the report)

Small hand-written evaluation set · one domain (Europarl) for training · Darija written by one author · timing depends on the machine · `tokens_per_word` uses whitespace words, not comparable across scripts (hence characters and bytes per token).